# SmolLM2-360M — DRAG Knowledge Distillation (GKD)

**Student**: `HuggingFaceTB/SmolLM2-360M-Instruct` (~0.7 GB fp16)  
**Teacher**: `meta-llama/Llama-3.1-8B-Instruct` (~16 GB fp16, already approved)  
**Method**: Generalized Knowledge Distillation (GKD) via TRL `GKDTrainer`  
**Target runtime**: H100 96 GB GPU (~1–2 hours)

## Why 8B instead of 70B?
Llama 3.3 70B with 4-bit NF4 (`bitsandbytes`) OOMs on H100 because `bitsandbytes`
materializes the full bf16 weights (~140 GB) in-place before quantizing, consuming all
96 GB VRAM before the student model can even load.

**Llama 3.1 8B is the right teacher here:**
- 22× size ratio (8B → 360M) is within the optimal KD range (studies show 5–30× works best)
- Loads entirely in fp16 (~16 GB), leaving ~78 GB free for student + optimizer + activations
- **True token-level KL divergence is preserved** — this is what makes GKD better than plain SFT
- Llama 3.1 8B is instruction-tuned and follows concise-answer prompts reliably
- Already approved: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

## VRAM budget
| Component | Size |
|---|---|
| Llama 3.1 8B (fp16) | ~16 GB |
| SmolLM2-360M (bf16 training) | ~0.7 GB |
| Optimizer states (AdamW) | ~1.4 GB |
| Activations + KV cache | ~5 GB |
| **Total** | **~23 GB** (72 GB headroom on H100) |

## What makes this real distillation
The teacher is loaded alongside the student and generates **soft-label logits live during
each training step**. The student minimizes:
- **KD loss** (λ): forward KL divergence between student and teacher token distributions
- **SFT loss** (1-λ): cross-entropy on original curated answers

Vocabulary mismatch (Llama ≠ SmolLM2) is handled at the sequence level:
teacher generates text → decoded to string → re-tokenized by student tokenizer.

## Prerequisites
- Llama 3.1 approved at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct ✅
- HF token with read access in Cell 2
- Google Drive mounted — data at `MyDrive/ColabDrive/Portfolio/`
  - `training_data.jsonl` — 96 original SFT examples
  - `training_data_drag.jsonl` — ~400 DRAG-format examples (run `npm run gen:all-training` locally first)

## Steps
1. Run `npm run gen:all-training` locally to generate triples + DRAG data
2. Upload both JSONL files to Drive at `MyDrive/ColabDrive/Portfolio/`
3. Run Cell 1 (install)
4. Run Cell 2 (HF login)
5. Run Cell 3 (mount Google Drive)
6. Run remaining cells in order
7. Checkpoint saves to `MyDrive/ColabDrive/Portfolio/checkpoint_kd/`
8. ONNX saves to `MyDrive/ColabDrive/Portfolio/onnx-export/`

# Developer Notes

## 1) The "Vocabulary Wall" (Showstopper)

This was the most critical finding. We initially tried `GKDTrainer` (Generalized Knowledge Distillation), which typically compares teacher and student probability distributions (logits).

- **Error:** `RuntimeError: stack expects each tensor to be equal size`
- **Lesson:** Standard online logit-based distillation does not work between Llama 3 (128k vocab) and SmolLM2 (49k vocab). Their output vectors have different lengths and cannot be compared 1:1.
- **Fix:** Switched to **Offline Distillation** (Cell 7). The teacher generates answers, and the student trains on those outputs. This works regardless of tokenizer mismatch.

## 2) Library "Bleeding Edge" Pains

We needed a very new TRL feature (`GKDTrainer`).

- **Missing imports:** Even with `pip install trl>=0.12.0`, the class was unavailable; installed from source (`git+github...`).
- **Hidden path:** Class was not in the main module; found under `trl.experimental.gkd`.
- **Unstable APIs:** Arguments changed; `max_seq_length` was rejected by both config and trainer, so defaults were used.

## 3) Dependency Hell (Training vs. Export)

- **Conflict:** Training stack (latest `transformers`, `trl`) conflicted with export stack (`optimum-onnx` preferring older `transformers`).
- **Solution:** Separated phases. Keep training dependencies in Cell 1; install export tooling in Cell 10 after training.

In [ ]:
# Cell 1 — Install dependencies
# Uninstall conflicting packages first to ensure a clean slate
!pip uninstall -y optimum-onnx optimum trl

# GKDTrainer is in trl>=0.12.0. We install from source to be safe.
!pip install -q git+https://github.com/huggingface/trl.git

# Install main libraries
!pip install -q -U transformers datasets accelerate bitsandbytes huggingface_hub

# For ONNX export, install optimum and onnxruntime.
# We avoid 'optimum[onnx]' package which has strict pinning and use main 'optimum' instead.
!pip install -q optimum onnx onnxruntime

print('Installation complete.')

In [ ]:
# Cell 2 — HF login (for uploading checkpoint to Hub)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Cell 3 — Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/ColabDrive/Portfolio'
# Both datasets: original SFT Q&A + new DRAG-format (system+evidence+triples)
DATA_PATHS   = [
    f'{DRIVE_ROOT}/training_data.jsonl',       # 96 original SFT examples
    f'{DRIVE_ROOT}/training_data_drag.jsonl',  # ~400 DRAG-format examples
]
CKPT_DIR     = f'{DRIVE_ROOT}/checkpoint_kd'    # separate from SFT checkpoint
ONNX_DIR     = f'{DRIVE_ROOT}/onnx-export'

import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)

print(f'Checkpoint: {CKPT_DIR}')
print(f'ONNX:       {ONNX_DIR}')
for p in DATA_PATHS:
    print(f'  {"✅" if os.path.exists(p) else "❌"} {p}')

In [ ]:
# Cell 4 — Load and combine both datasets
# training_data.jsonl       — 96 original SFT pairs (user/assistant only)
# training_data_drag.jsonl  — ~400 DRAG-format pairs (system+evidence+triples → answer)
# Combining both gives the student: direct Q&A recall + grounded evidence reasoning.
import json
from datasets import Dataset

rows = []
for path in DATA_PATHS:
    if not os.path.exists(path):
        print(f'⚠️  Skipping missing file: {path}')
        continue
    before = len(rows)
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            msgs = ex['messages']
            rows.append({'messages': msgs})
    print(f'Loaded {len(rows) - before} examples from {os.path.basename(path)}')

print(f'\nTotal examples: {len(rows)}')
dataset = Dataset.from_list(rows)

# 90/10 train/eval split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds  = split['test']

print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')

In [ ]:
# Cell 5 — Load teacher (Llama 3.1 8B) and student (SmolLM2-360M)
#
# Teacher in fp16 uses ~16 GB — leaves 78+ GB free for student + optimizer.
# No quantization needed; fp16 gives full logit fidelity for KD.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

TEACHER_ID = 'meta-llama/Llama-3.1-8B-Instruct'
STUDENT_ID = 'HuggingFaceTB/SmolLM2-360M-Instruct'

print('Loading teacher tokenizer...')
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)

print('Loading teacher model (Llama 3.1 8B, fp16)...')
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
teacher_model.eval()

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
print(f'Teacher loaded. VRAM allocated: {allocated:.1f} GB | reserved: {reserved:.1f} GB')

print('Loading student model (bf16 for training stability + H100 efficiency)...')
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
student_model.train()

allocated2 = torch.cuda.memory_allocated() / 1e9
free_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) - allocated2
print(f'Both loaded. VRAM used: {allocated2:.1f} GB | free: {free_gb:.1f} GB')
print(f'Teacher params: {sum(p.numel() for p in teacher_model.parameters()) / 1e9:.1f}B')
print(f'Student params: {sum(p.numel() for p in student_model.parameters()) / 1e6:.0f}M')

In [ ]:
# Cell 6 — Format dataset as chat-templated text (for GKDTrainer prompt/completion split)
# GKDTrainer expects 'prompt' (user turn only) so the teacher can generate the completion.

def format_prompt(example):
    msgs = example['messages']
    # Prompt = everything except the last assistant turn
    prompt_msgs = [m for m in msgs if m['role'] != 'assistant']
    prompt = student_tokenizer.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Teacher-formatted prompt — Llama uses its own chat template, not ChatML
    teacher_prompt = teacher_tokenizer.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Completion = the original assistant answer (used as SFT reference)
    completion = next((m['content'] for m in reversed(msgs) if m['role'] == 'assistant'), '')
    return {'prompt': prompt, 'teacher_prompt': teacher_prompt, 'completion': completion}

train_ds = train_ds.map(format_prompt)
eval_ds  = eval_ds.map(format_prompt)

print('Sample prompt:')
print(train_ds[0]['prompt'])
print('Sample completion:')
print(train_ds[0]['completion'])

In [ ]:
# Cell 7a — Offline Distillation (Generate)
# PROBLEM: GKDTrainer requires identical vocabularies for JSD loss.
# SOLUTION: We switch to Offline Distillation (Forward KL).
#   1. Teacher generates synthetic answers in batches (faster, more VRAM-efficient).
#   2. Teacher is offloaded to CPU / deleted to free ~16 GB before SFT.
#   3. Student trains on teacher outputs with max batch size (128).

import gc
import torch
from tqdm.auto import tqdm
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# ── Step 1: Batched Teacher Generation ─────────────────────────────────────
print("\n--- Step 1: Generating synthetic data with Teacher (Llama 3.1 8B) ---")

TEACHER_GEN_BATCH = 8          # Batch teacher inference — 8 prompts at once
TEACHER_MAX_PROMPT_LEN = 1536  # Full evidence context (was 768 — caused truncated teacher input)
TEACHER_MAX_NEW_TOKENS = 512   # Max completion length

# Left-pad so all sequences in batch end at the same position (required for generation)
# Llama has no pad token by default — use EOS (masked by attention_mask during generation)
teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
teacher_tokenizer.padding_side = 'left'

prompts    = train_ds['teacher_prompt']  # Llama-formatted — teacher sees its own template
all_msgs   = train_ds['messages']         # original message lists — used to rebuild for student
synthetic_rows = []

teacher_model.eval()
print(f"Generating responses for {len(prompts)} examples "
      f"(batch={TEACHER_GEN_BATCH}, max_new={TEACHER_MAX_NEW_TOKENS})...")

for i in tqdm(range(0, len(prompts), TEACHER_GEN_BATCH)):
    batch_prompts = list(prompts[i : i + TEACHER_GEN_BATCH])

    enc = teacher_tokenizer(
        batch_prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=TEACHER_MAX_PROMPT_LEN,
        add_special_tokens=False,
    ).to(teacher_model.device)

    with torch.no_grad():
        out_ids = teacher_model.generate(
            **enc,
            max_new_tokens=TEACHER_MAX_NEW_TOKENS,
            do_sample=False,          # Greedy — deterministic targets, no label noise
            pad_token_id=teacher_tokenizer.eos_token_id,
        )

    # All items in batch share the same padded input length — strip it off
    new_start = enc.input_ids.shape[1]
    for j, out in enumerate(out_ids):
        generated_text = teacher_tokenizer.decode(
            out[new_start:], skip_special_tokens=True
        ).strip()
        orig_msgs   = all_msgs[i + j]
        prompt_msgs = [m for m in orig_msgs if m['role'] != 'assistant']
        full_msgs   = prompt_msgs + [{'role': 'assistant', 'content': generated_text}]
        synthetic_rows.append({'messages': full_msgs})

    # Return GPU tensors immediately — keeps VRAM flat
    del enc, out_ids
    torch.cuda.empty_cache()

synthetic_ds = Dataset.from_list(synthetic_rows)
print(f"Generated {len(synthetic_ds)} synthetic examples.")
print("Sample Teacher Output:", synthetic_ds[0]['messages'][-1]['content'][:200])

# ── Offload teacher before SFT to free ~16 GB VRAM ─────────────────────────
print("\nOffloading teacher to CPU to free VRAM for SFT...")
teacher_model.cpu()
del teacher_model
gc.collect()
torch.cuda.empty_cache()

free_gb = (
    torch.cuda.get_device_properties(0).total_memory
    - torch.cuda.memory_allocated()
) / 1e9
print(f"VRAM after teacher offload: {free_gb:.1f} GB free")


In [ ]:
# Cell 7b — Offline Distillation (SFT)
# completion_only_loss=True masks all prompt tokens — loss on completions only.
# synthetic_ds already has 'prompt' and 'completion' columns from Cell 7a.

print("\n--- Step 2: Training Student (SmolLM2-360M) ---")

# Diagnostic: verify completion_only_loss is a valid SFTConfig field
import inspect
valid_params = inspect.signature(SFTConfig.__init__).parameters
print("completion_only_loss supported:", "completion_only_loss" in valid_params)

# Diagnostic: inspect synthetic_ds
print(f"synthetic_ds size: {len(synthetic_ds)}")
print(f"Columns: {synthetic_ds.column_names}")
print(f"Sample assistant turn: {synthetic_ds[0]['messages'][-1]['content'][:120]}")

sft_config = SFTConfig(
    output_dir=CKPT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=32,   # teacher offloaded — ~80 GB free
    gradient_accumulation_steps=1,
    learning_rate=3e-6,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    max_grad_norm=0.3,
    bf16=True,
    logging_steps=5,
    report_to='none',
    save_strategy='epoch',
    eval_strategy='no',
    max_seq_length=2048,   # SmolLM2-360M's full context window (was 1024 — truncated all examples)
    completion_only_loss=True,        # key fix: prompt tokens excluded from loss
)

trainer = SFTTrainer(
    model=student_model,
    args=sft_config,
    train_dataset=synthetic_ds,       # switch to train_ds if synthetic completions don't converge
    processing_class=student_tokenizer,
)

print("Starting SFT (completion-only loss — prompt tokens masked)...")
trainer.train()
print("Distillation (SFT) complete.")


In [ ]:
# Cell 8 — Quick qualitative eval
import torch
from transformers import GenerationConfig

model = trainer.model
model.eval()

# Reset generation config to avoid any max_length=20 artifact saved during SFT
model.generation_config = GenerationConfig(
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=student_tokenizer.eos_token_id,
    eos_token_id=student_tokenizer.eos_token_id,
)

system = 'You are a helpful assistant for Kham\'s portfolio. Be concise and accurate.'

prompts = [
    'What are Kham\'s main technical skills?',
    'How many years of experience does Kham have in AI/ML?',
    'What companies has Kham worked at?',
    'Tell me about Kham\'s DRAG project.',
]

print("=== Fine-tuned SmolLM2-360M Qualitative Eval ===\n")
for q in prompts:
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': q}]
    text = student_tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = student_tokenizer(text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                                  pad_token_id=student_tokenizer.eos_token_id)

    # Decode only the newly generated tokens
    new_tokens = out_ids[0][input_len:]
    answer = student_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print(f"Q: {q}")
    print(f"A: {answer if answer else '[EMPTY — model generated EOS immediately]'}")
    print()

In [ ]:
# Cell 9 — Save checkpoint to Drive and push to HF Hub
import os

trainer.save_model(CKPT_DIR)
student_tokenizer.save_pretrained(CKPT_DIR)
print(f'Checkpoint saved to Drive: {CKPT_DIR}')

# Push to HF Hub
HF_REPO = '<your-hf-username>/smollm2-drag'
trainer.model.push_to_hub(HF_REPO, commit_message='GKD distillation checkpoint (Llama 3.3 70B teacher)')
student_tokenizer.push_to_hub(HF_REPO)
print(f'Pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# Cell 10 — Export to ONNX (fp32)
# WORKING COMMAND (confirmed): --dtype fp32 --opset 18
# NOTE: --int8 flag is NOT supported by optimum-cli in this environment.
# NOTE: quantize_dynamic (Cell 11) produces float16 scale tensors that crash ONNX Runtime Web/WASM.
# RECOMMENDATION: Use fp32 export directly. transformers.js dtype: fp32 loads model.onnx correctly.

!pip install -q "optimum[onnx]" onnx onnxruntime

import os
os.makedirs(ONNX_DIR, exist_ok=True)
print(f"Exporting model from {CKPT_DIR} to {ONNX_DIR}...")

!optimum-cli export onnx \
    --model "{CKPT_DIR}" \
    --task text-generation-with-past \
    --device cuda \
    --dtype fp32 \
    --opset 18 \
    "{ONNX_DIR}"

print("ONNX export complete. Files:", os.listdir(ONNX_DIR))

In [ ]:
# (This cell was a scratch replacement — superseded by Cell 10 above.)
# No action needed here.

In [ ]:
# Cell 11 — Quantize ONNX  *** SKIP THIS CELL ***
#
# All tested quantization paths produce WASM-incompatible output:
#   - quantize_dynamic(per_channel=True)  → tensor(float16) scale error in ONNX Runtime Web
#   - quantize_dynamic(per_channel=False) → same float16 scale issue on embedding-adjacent nodes
#   - ORTQuantizer from optimum           → ImportError: huggingface_hub version conflict in Colab
#   - optimum-cli --int8                  → "unrecognized arguments: --int8"
#
# SOLUTION: Use the fp32 model.onnx directly (uploaded in Cell 12).
#           In the browser: transformers.js dtype: "fp32" loads model.onnx correctly.
#           The model is ~784 MB but cached after first download.

print("Cell 11 SKIPPED — fp32 model.onnx is used directly (see Cell 12).")

In [ ]:
# import os
# from onnxruntime.quantization import quantize_dynamic, QuantType

# # Loop through ALL exported onnx files in your directory
# for root, dirs, files in os.walk(ONNX_DIR):
#     for f in files:
#         if f.endswith(".onnx") and not f.endswith("_quantized.onnx"):
#             input_path = os.path.join(root, f)
#             output_path = input_path.replace(".onnx", "_quantized.onnx")
            
#             print(f"Quantizing: {f}...")
#             quantize_dynamic(
#                 model_input=input_path,
#                 model_output=output_path,
#                 per_channel=True,      # Better accuracy for Transformers
#                 reduce_range=True,     # Necessary for older CPUs/WASM
#                 weight_type=QuantType.QInt8
#             )

# print("All components quantized. Use the '_quantized.onnx' files in your browser project.")


In [ ]:
# Cell 12 — Upload ONNX to HF Hub
# Uploads the entire ONNX_DIR (model.onnx + model_with_past.onnx + config files).
# transformers.js loads model.onnx with dtype: "fp32" — no quantized file needed.

import os
from huggingface_hub import HfApi

api = HfApi()
HF_REPO = "<your-hf-username>/smollm2-drag"

print(f"Uploading folder {ONNX_DIR} to {HF_REPO}...")
api.upload_folder(
    folder_path=ONNX_DIR,
    repo_id=HF_REPO,
    repo_type="model",
    commit_message="Add fp32 ONNX export (opset 18, offline KD from Llama 3.1 8B)",
)
print(f"Done! https://huggingface.co/{HF_REPO}/tree/main")